# Main

In [16]:
import helpers.solver as solver
import helpers.postprocess as postprocess
import numpy as np

## Truss Validation

### Model Definition

The node numbering order and corresponding positions are as follows:

- **Odd-numbered nodes** are on the left truss plane: $x=0$
- **Even-numbered nodes** are on the right truss plane: $x=3$
- Along the roof truss length, the nodes are numbered in ascending order of **$y$**
- The roof height is represented by the **$z$ coordinate**: bottom chord nodes have $z=0$, while upper chord or ridge nodes have $z=2.598$ or $z=5.196$

| Node | Position $(X, Y, Z)$ m | Description |
|---|---|---|
| 1 | $(0,\ 0,\ 0)$ | Left end bottom node |
| 2 | $(3,\ 0,\ 0)$ | Right end bottom node |
| 3 | $(0,\ 4,\ 2.598)$ | Left front upper chord node |
| 4 | $(3,\ 4.5,\ 2.598)$ | Right front upper chord node |
| 5 | $(0,\ 6,\ 0)$ | Left bottom chord node |
| 6 | $(3,\ 6,\ 0)$ | Right bottom chord node |
| 7 | $(0,\ 9,\ 5.196)$ | Left node near the ridge |
| 8 | $(3,\ 9,\ 5.196)$ | Right node near the ridge |
| 9 | $(0,\ 12,\ 0)$ | Left bottom chord node |
| 10 | $(3,\ 12,\ 0)$ | Right bottom chord node |
| 11 | $(0,\ 13.5,\ 2.598)$ | Left rear upper chord node |
| 12 | $(3,\ 13.5,\ 2.598)$ | Right rear upper chord node |
| 13 | $(0,\ 18,\ 0)$ | Left end bottom node |
| 14 | $(3,\ 18,\ 0)$ | Right end bottom node |

It can also be understood in pairs of transverse nodes:

- Pair 1: 1–2, $y=0$
- Pair 2: 3–4, $y=4.5$
- Pair 3: 5–6, $y=6$
- Pair 4: 7–8, $y=9$
- Pair 5: 9–10, $y=12$
- Pair 6: 11–12, $y=13.5$
- Pair 7: 13–14, $y=18$

The element definitions are listed below.

All members are defined as **TRUSS** elements with **Material 1**, **Section 1**, and **MT = 0**.

| Element | Connectivity $(i, j)$ | Description |
|---|---:|---|
| 1 | (1, 5) | Left bottom chord member |
| 2 | (1, 3) | Left end diagonal member |
| 3 | (3, 5) | Left lower diagonal member |
| 4 | (3, 7) | Left upper chord member |
| 5 | (5, 7) | Left web diagonal member |
| 6 | (5, 9) | Left bottom chord member |
| 7 | (7, 9) | Left web diagonal member |
| 8 | (7, 11) | Left upper chord member |
| 9 | (9, 11) | Left lower diagonal member |
| 10 | (9, 13) | Left bottom chord member |
| 11 | (11, 13) | Left end diagonal member |
| 12 | (2, 6) | Right bottom chord member |
| 13 | (2, 4) | Right end diagonal member |
| 14 | (4, 6) | Right lower diagonal member |
| 15 | (4, 8) | Right upper chord member |
| 16 | (6, 8) | Right web diagonal member |
| 17 | (6, 10) | Right bottom chord member |
| 18 | (8, 10) | Right web diagonal member |
| 19 | (8, 12) | Right upper chord member |
| 20 | (10, 12) | Right lower diagonal member |
| 21 | (10, 14) | Right bottom chord member |
| 22 | (12, 14) | Right end diagonal member |
| 23 | (1, 2) | Transverse end bottom tie |
| 24 | (3, 4) | Transverse front upper tie |
| 25 | (5, 6) | Transverse bottom tie |
| 26 | (7, 8) | Transverse ridge tie |
| 27 | (9, 10) | Transverse bottom tie |
| 28 | (11, 12) | Transverse rear upper tie |
| 29 | (13, 14) | Transverse end bottom tie |
| 30 | (1, 4) | Front end transverse diagonal |
| 31 | (2, 3) | Front end transverse diagonal |
| 32 | (1, 6) | Lower transverse diagonal |
| 33 | (2, 5) | Lower transverse diagonal |
| 34 | (3, 8) | Front upper transverse diagonal |
| 35 | (4, 7) | Front upper transverse diagonal |
| 36 | (5, 10) | Middle lower transverse diagonal |
| 37 | (6, 9) | Middle lower transverse diagonal |
| 38 | (7, 12) | Upper transverse diagonal |
| 39 | (8, 11) | Upper transverse diagonal |
| 40 | (9, 14) | Rear lower transverse diagonal |
| 41 | (10, 13) | Rear lower transverse diagonal |
| 42 | (11, 14) | Rear end transverse diagonal |
| 43 | (12, 13) | Rear end transverse diagonal |

It can also be understood by member groups:

- **Left truss plane members:** 1--11  
- **Right truss plane members:** 12--22  
- **Transverse members:** 23--29  
- **Transverse bracing members:** 30--43

In addition, the cross-section info are listed below:

- Young's Modulus: $E=70 GPa$
- Area: $A=4000 mm^2$

In summary:

**The nodes are numbered from front to back in the $y$ direction, and for each transverse section, the left node is numbered first and the right node second. Thus, left-side nodes are odd and right-side nodes are even.**

---

### Validation File Description

In addition to the geometric modeling information above, four separate validation files were prepared to verify the implementation of different load cases. All four files use the same structural geometry, element connectivity, material, section, and boundary-condition definitions. The difference between them lies in the applied loading or imposed action.

#### 1. `validation_truss_nodal_loads.json`

This file is used to validate the nodal load function.

- A nodal load of $F_z = -100$ $kN$ is applied at node 11
- A nodal load of $F_x = 100$ $kN$ is applied at node 12
- All other nodal loads are zero
- No temperature load, fabrication error, or support displacement is applied in this case

This case is intended to verify whether the program can correctly assemble concentrated nodal forces into the global load vector and produce the corresponding structural response.

#### 2. `validation_truss_supp_disp.json`

This file is used to validate the support displacement function.

- An imposed support displacement of $u_z = 0.01$ $m$ is specified
- No external nodal load is applied
- No temperature load or fabrication error is applied in this case

This case is intended to verify whether the program can correctly convert prescribed support movement into the equivalent structural response and reaction output.

#### 3. `validation_truss_temperature.json`

This file is used to validate the temperature load function.

- A temperature change of $\Delta T = 30^\circ\mathrm{C}$ is assigned to all 43 truss elements
- No external nodal load is applied
- No support displacement or fabrication error is applied in this case

This case is intended to verify whether the program can correctly calculate thermal strain effects and assemble the corresponding equivalent load contribution.

#### 4. `validation_truss_fab_error.json`

This file is used to validate the fabrication error function.

- A fabrication error value of $\mathrm{ERROR} = 0.01$ $m$ on element 1 is assigned
- No external nodal load is applied
- No support displacement or temperature load is applied in this case

This case is intended to verify whether the program can correctly account for initial length error effects in truss members and include them in the analysis.

#### Summary

These four validation files were created so that each loading function can be checked independently before applying the program to more complicated structural cases. In other words:

- `validation_truss_nodal_loads.json` verifies nodal force input
- `validation_truss_supp_disp.json` verifies imposed support displacement
- `validation_truss_temperature.json` verifies temperature loading
- `validation_truss_fab_error.json` verifies fabrication error loading

By separating the validation cases in this way, it becomes easier to confirm that each loading module is implemented correctly and that the corresponding response is physically reasonable.

### Validation Nodal Loads

In [17]:
filepath = 'inputs/validation_truss_nodal_loads.json'

result = solver.truss_solver(filepath)
pp = postprocess.truss_output(filepath, result)


,node,x (M),y (M),z (M),ux (M),uy (M),uz (M),rx (rad),ry (rad),rz (rad),|u| (M)
0,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-0.000003,0.000001,-0.000006,0.000000
1,2.000000,3.000000,0.000000,0.000000,0.000022,0.000000,0.000000,-0.000002,0.000000,-0.000003,0.000022
2,3.000000,0.000000,4.500000,2.598076,0.000058,0.000041,-0.000003,-0.000003,0.000002,-0.000015,0.000071
3,4.000000,3.000000,4.500000,2.598076,0.000064,-0.000005,-0.000008,-0.000002,0.000001,-0.000015,0.000064
4,5.000000,0.000000,6.000000,0.000000,0.000080,0.000040,-0.000012,-0.000001,0.000001,-0.000017,0.000090
5,6.000000,3.000000,6.000000,0.000000,0.000086,-0.000009,-0.000013,-0.000001,0.000000,-0.000017,0.000087
6,7.000000,0.000000,9.000000,5.196152,0.000141,0.000052,-0.000013,-0.000001,0.000000,-0.000020,0.000151
7,8.000000,3.000000,9.000000,5.196152,0.000144,-0.000008,-0.000015,-0.000001,0.000000,-0.000020,0.000145
8,9.000000,0.000000,12.000000,0.000000,0.000204,0.000059,-0.000014,0.000001,0.000000,-0.000022,0.000213
9,10.000000,3.000000,12.000000,0.000000,0.000205,-0.000007,-0.000015,0.000001,0.000001,-0.000022,0.000205


,metric,node,value (M),ux (M),uy (M),uz (M)
0,max |u|,13.000000,0.000342,0.000338,0.000049,0.000000
1,max |ux|,13.000000,0.000338,0.000338,0.000049,0.000000
2,max |uy|,9.000000,0.000059,0.000204,0.000059,-0.000014
3,max |uz|,11.000000,-0.000019,0.000237,0.000054,-0.000019


,node,Fx (),Fy (),Fz ()
0,1.000000,-100.000000,-450.000000,-23.689030
1,2.000000,nan,450.000000,48.689030
2,13.000000,nan,nan,37.086490
3,14.000000,nan,nan,37.913510


,element,type,i,j,L (M),Δu_local_x (M),axial strain,N_tension (+) (KN),sigma (KN/M^2)
0,1.000000,T,1.000000,5.000000,6.000000,0.000040,0.000007,1.889223,472.305748
1,2.000000,T,1.000000,3.000000,5.196152,0.000034,0.000007,1.827424,456.856035
2,3.000000,T,3.000000,5.000000,3.000000,0.000008,0.000003,0.723190,180.797474
3,4.000000,T,3.000000,7.000000,5.196152,0.000004,0.000001,0.232643,58.160754
4,5.000000,T,5.000000,7.000000,6.000000,0.000005,0.000001,0.223128,55.782112
5,6.000000,T,5.000000,9.000000,6.000000,0.000018,0.000003,0.858028,214.507101
6,7.000000,T,7.000000,9.000000,6.000000,0.000004,0.000001,0.208258,52.064598
7,8.000000,T,7.000000,11.000000,5.196152,0.000005,0.000001,0.276900,69.224904
8,9.000000,T,9.000000,11.000000,3.000000,-0.000006,-0.000002,-0.563415,-140.853655
9,10.000000,T,9.000000,13.000000,6.000000,-0.000010,-0.000002,-0.446608,-111.651890


,element,i,j,L (M),N_tension (+) (KN),sigma (KN/M^2),u_g_1 (M),u_g_2 (M),u_g_3 (M),u_g_4 (rad),u_g_5 (rad),u_g_6 (rad),u_g_7 (M),u_g_8 (M),u_g_9 (M),u_g_10 (rad),u_g_11 (rad),u_g_12 (rad),u_l_1 (M),u_l_2 (M),u_l_3 (M),u_l_4 (rad),u_l_5 (rad),u_l_6 (rad),u_l_7 (M),u_l_8 (M),u_l_9 (M),u_l_10 (rad),u_l_11 (rad),u_l_12 (rad),q_l_1 (KN),q_l_2 (KN),q_l_3 (KN),q_l_4 (KN*M),q_l_5 (KN*M),q_l_6 (KN*M),q_l_7 (KN),q_l_8 (KN),q_l_9 (KN),q_l_10 (KN*M),q_l_11 (KN*M),q_l_12 (KN*M)
0,1.000000,1.000000,5.000000,6.000000,1.889223,472.305748,0.000000,0.000000,0.000000,-0.000003,0.000001,-0.000006,0.000080,0.000040,-0.000012,-0.000001,0.000001,-0.000017,0.000000,0.000000,0.000000,0.000001,0.000003,-0.000006,0.000040,-0.000080,-0.000012,0.000001,0.000001,-0.000017,-1.889223,39.334505,-7.311756,2.269554,45.039780,249.142842,1.889223,-39.334505,7.311756,-2.269554,-1.169241,-13.135810
1,2.000000,1.000000,3.000000,5.196152,1.827424,456.856035,0.000000,0.000000,0.000000,-0.000003,0.000001,-0.000006,0.000058,0.000041,-0.000003,-0.000003,0.000002,-0.000015,0.000000,0.000000,0.000000,-0.000002,0.000006,0.000003,0.000034,0.000023,-0.000058,-0.000006,0.000014,0.000003,-1.827424,-42.591867,36.671449,22.715021,-209.055285,-103.429215,1.827424,42.591867,-36.671449,-22.715021,18.504846,-117.884619
2,3.000000,3.000000,5.000000,3.000000,0.723190,180.797474,0.000058,0.000041,-0.000003,-0.000003,0.000002,-0.000015,0.000080,0.000040,-0.000012,-0.000001,0.000001,-0.000017,0.000023,0.000034,0.000058,0.000014,-0.000006,-0.000003,0.000031,0.000029,0.000080,0.000015,-0.000008,-0.000001,-0.723190,-39.889113,-3.253490,-9.294723,43.946834,-93.523943,0.723190,39.889113,3.253490,9.294723,-34.186364,-26.143396
3,4.000000,3.000000,7.000000,5.196152,0.232643,58.160754,0.000058,0.000041,-0.000003,-0.000003,0.000002,-0.000015,0.000141,0.000052,-0.000013,-0.000001,0.000000,-0.000020,0.000034,0.000023,-0.000058,-0.000006,0.000014,0.000003,0.000038,0.000037,-0.000141,-0.000010,0.000017,0.000001,-0.232643,-22.092309,8.444520,17.014520,-66.755577,-36.665872,0.232643,22.092309,-8.444520,-17.014520,22.876563,-78.129135
4,5.000000,5.000000,7.000000,6.000000,0.223128,55.782112,0.000080,0.000040,-0.000012,-0.000001,0.000001,-0.000017,0.000141,0.000052,-0.000013,-0.000001,0.000000,-0.000020,0.000010,0.000041,-0.000080,-0.000015,0.000009,0.000001,0.000015,0.000051,-0.000141,-0.000017,0.000010,0.000001,-0.223128,-9.355151,13.769393,11.394745,-55.067065,-26.956470,0.223128,9.355151,-13.769393,-11.394745,-27.549291,-29.174436
5,6.000000,5.000000,9.000000,6.000000,0.858028,214.507101,0.000080,0.000040,-0.000012,-0.000001,0.000001,-0.000017,0.000204,0.000059,-0.000014,0.000001,0.000000,-0.000022,0.000040,-0.000080,-0.000012,0.000001,0.000001,-0.000017,0.000059,-0.000204,-0.000014,0.000000,-0.000001,-0.000022,-0.858028,22.582809,4.052077,2.366596,15.627189,127.352237,0.858028,-22.582809,-4.052077,-2.366596,-39.939648,8.144617
6,7.000000,7.000000,9.000000,6.000000,0.208258,52.064598,0.000141,0.000052,-0.000013,-0.000001,0.000000,-0.000020,0.000204,0.000059,-0.000014,0.000001,0.000000,-0.000022,0.000037,0.000038,0.000141,0.000017,-0.000010,-0.000001,0.000042,0.000044,0.000204,0.000019,-0.000011,0.000001,-0.208258,-24.670143,-0.749932,-8.389663,18.602767,-100.684864,0.208258,24.670143,0.749932,8.389663,-14.103175,-47.335993
7,8.000000,7.000000,11.000000,5.196152,0.276900,69.224904,0.000141,0.000052,-0.000013,-0.000001,0.000000,-0.000020,0.000237,0.000054,-0.000019,0.000001,-0.000001,-0.000022,0.000051,0.000015,0.000141,0.000010,-0.000017,-0.000001,0.000056,0.000011,0.000237,0.000010,-0.000019,0.000001,-0.276900,11.448421,-3.418951,0.381995,39.533564,3.412357,0.276900,-11.448421,3.418951,-0.381995,-21.768174,56.075383
8,9.000000,9.000000,11.000000,3.000000,-0.563415,-140.853655,0.000204,0.000059,-0.000014,0.000001,0.000000,-0.000022,0.000237,0.000054,-0.000019,0.000001,-0.000001,-0.000022,0.000017,0.000058,-0.000204,-0.000019,0.000011,-0.000001,0.000011,0.000056,-0.000237,-0.000019,0.000010,-0.000001,0.

### Validation Support Displacement

In [18]:
filepath = 'inputs/validation_truss_supp_disp.json'

result = solver.truss_solver(filepath)
pp = postprocess.truss_output(filepath, result)

,node,x (M),y (M),z (M),ux (M),uy (M),uz (M),rx (rad),ry (rad),rz (rad),|u| (M)
0,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.010000,-0.000509,0.003109,0.000014,0.010000
1,2.000000,3.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-0.000046,0.003109,0.000014,0.000000
2,3.000000,0.000000,4.500000,2.598076,0.006211,0.000615,0.007104,-0.000345,0.002182,0.000122,0.009456
3,4.000000,3.000000,4.500000,2.598076,0.006211,0.000829,0.000396,-0.000211,0.002182,0.000122,0.006278
4,5.000000,0.000000,6.000000,0.000000,0.000742,0.000048,0.006382,-0.000348,0.002036,-0.000110,0.006425
5,6.000000,3.000000,6.000000,0.000000,0.000742,-0.000048,0.000285,-0.000208,0.002036,-0.000110,0.000796
6,7.000000,0.000000,9.000000,5.196152,0.010068,0.001143,0.005000,-0.000318,0.001667,0.000211,0.011299
7,8.000000,3.000000,9.000000,5.196152,0.010068,0.001744,-0.000000,-0.000238,0.001667,0.000211,0.010218
8,9.000000,0.000000,12.000000,0.000000,0.002073,0.000048,0.003618,-0.000348,0.001298,-0.000110,0.004170
9,10.000000,3.000000,12.000000,0.000000,0.002073,-0.000048,-0.000285,-0.000208,0.001298,-0.000110,0.002093


,metric,node,value (M),ux (M),uy (M),uz (M)
0,max |u|,7.000000,0.011299,0.010068,0.001143,0.005000
1,max |ux|,7.000000,0.010068,0.010068,0.001143,0.005000
2,max |uy|,8.000000,0.001744,0.010068,0.001744,-0.000000
3,max |uz|,1.000000,0.010000,0.000000,0.000000,0.010000


,node,Fx (),Fy (),Fz ()
0,1.000000,0.000000,0.000000,27173.533449
1,2.000000,nan,-0.000000,-27173.533449
2,13.000000,nan,nan,-27173.533449
3,14.000000,nan,nan,27173.533449


,element,type,i,j,L (M),Δu_local_x (M),axial strain,N_tension (+) (KN),sigma (KN/M^2)
0,1.000000,T,1.000000,5.000000,6.000000,0.000048,0.000008,2.236757,559.189130
1,2.000000,T,1.000000,3.000000,5.196152,-0.000915,-0.000176,-49.329788,-12332.446912
2,3.000000,T,3.000000,5.000000,3.000000,0.000343,0.000114,31.975156,7993.789001
3,4.000000,T,3.000000,7.000000,5.196152,-0.000595,-0.000114,-32.040724,-8010.180967
4,5.000000,T,5.000000,7.000000,6.000000,-0.000649,-0.000108,-30.281491,-7570.372639
5,6.000000,T,5.000000,9.000000,6.000000,0.000000,0.000000,0.000000,0.000000
6,7.000000,T,7.000000,9.000000,6.000000,0.000649,0.000108,30.281491,7570.372639
7,8.000000,T,7.000000,11.000000,5.196152,0.000595,0.000114,32.040724,8010.180967
8,9.000000,T,9.000000,11.000000,3.000000,-0.000343,-0.000114,-31.975156,-7993.789001
9,10.000000,T,9.000000,13.000000,6.000000,-0.000048,-0.000008,-2.236757,-559.189130


,element,i,j,L (M),N_tension (+) (KN),sigma (KN/M^2),u_g_1 (M),u_g_2 (M),u_g_3 (M),u_g_4 (rad),u_g_5 (rad),u_g_6 (rad),u_g_7 (M),u_g_8 (M),u_g_9 (M),u_g_10 (rad),u_g_11 (rad),u_g_12 (rad),u_l_1 (M),u_l_2 (M),u_l_3 (M),u_l_4 (rad),u_l_5 (rad),u_l_6 (rad),u_l_7 (M),u_l_8 (M),u_l_9 (M),u_l_10 (rad),u_l_11 (rad),u_l_12 (rad),q_l_1 (KN),q_l_2 (KN),q_l_3 (KN),q_l_4 (KN*M),q_l_5 (KN*M),q_l_6 (KN*M),q_l_7 (KN),q_l_8 (KN),q_l_9 (KN),q_l_10 (KN*M),q_l_11 (KN*M),q_l_12 (KN*M)
0,1.000000,1.000000,5.000000,6.000000,2.236757,559.189130,0.000000,0.000000,0.010000,-0.000509,0.003109,0.000014,0.000742,0.000048,0.006382,-0.000348,0.002036,-0.000110,0.000000,0.000000,0.010000,0.003109,0.000509,0.000014,0.000048,-0.000742,0.006382,0.002036,0.000348,-0.000110,-2.236757,1765.306014,4071.960356,4815.060439,-10331.833093,6749.665783,2.236757,-1765.306014,-4071.960356,-4815.060439,-14099.929041,3842.170299
1,2.000000,1.000000,3.000000,5.196152,-49.329788,-12332.446912,0.000000,0.000000,0.010000,-0.000509,0.003109,0.000014,0.006211,0.000615,0.007104,-0.000345,0.002182,0.000122,0.005000,-0.008660,0.000000,0.002700,0.001542,0.000509,0.004085,-0.005845,-0.006211,0.001951,0.000985,0.000345,49.329788,-3565.004968,-2124.264692,3877.810105,13023.120041,-7049.020034,-49.329788,3565.004968,2124.264692,-3877.810105,-1985.116917,-11475.289166
2,3.000000,3.000000,5.000000,3.000000,31.975156,7993.789001,0.006211,0.000615,0.007104,-0.000345,0.002182,0.000122,0.000742,0.000048,0.006382,-0.000348,0.002036,-0.000110,-0.005845,0.004085,0.006211,0.000985,0.001951,-0.000345,-0.005503,0.003232,0.000742,0.001113,0.001708,-0.000348,-31.975156,-5817.593708,-613.882642,-1152.431646,6595.321223,-8661.224997,31.975156,5817.593708,613.882642,1152.431646,-4753.673296,-8791.556127
3,4.000000,3.000000,7.000000,5.196152,-32.040724,-8010.180967,0.006211,0.000615,0.007104,-0.000345,0.002182,0.000122,0.010068,0.001143,0.005000,-0.000318,0.001667,0.000211,0.004085,-0.005845,-0.006211,0.001951,0.000985,0.000345,0.003490,-0.003759,-0.010068,0.001549,0.000650,0.000318,32.040724,-2180.040163,-2344.061499,2082.929817,10600.661529,-5299.666069,-32.040724,2180.040163,2344.061499,-2082.929817,1579.439311,-6028.154905
4,5.000000,5.000000,7.000000,6.000000,-30.281491,-7570.372639,0.000742,0.000048,0.006382,-0.000348,0.002036,-0.000110,0.010068,0.001143,0.005000,-0.000318,0.001667,0.000211,0.005551,-0.003149,-0.000742,0.000922,0.001818,0.000348,0.004902,-0.001510,-0.010068,0.001016,0.001338,0.000318,30.281491,1392.876964,-551.662365,-422.172585,7261.797895,4526.658595,-30.281491,-1392.876964,551.662365,422.172585,-3951.823703,3830.603192
5,6.000000,5.000000,9.000000,6.000000,0.000000,0.000000,0.000742,0.000048,0.006382,-0.000348,0.002036,-0.000110,0.002073,0.000048,0.003618,-0.000348,0.001298,-0.000110,0.000048,-0.000742,0.006382,0.002036,0.000348,-0.000110,0.000048,-0.002073,0.003618,0.001298,0.000348,-0.000110,-0.000000,2602.302714,2629.852015,3312.935558,-7889.556046,7806.908143,0.000000,-2602.302714,-2629.852015,-3312.935558,-7889.556046,7806.908143
6,7.000000,7.000000,9.000000,6.000000,30.281491,7570.372639,0.010068,0.001143,0.005000,-0.000318,0.001667,0.000211,0.002073,0.000048,0.003618,-0.000348,0.001298,-0.000110,-0.003759,0.003490,0.010068,0.000650,0.001549,-0.000318,-0.003110,0.001851,0.002073,0.000744,0.001069,-0.000348,-30.281491,-1392.876964,551.662365,-422.172585,3951.823703,-3830.603192,30.281491,1392.876964,-551.662365,422.172585,-7261.797895,-4526.658595
7,8.000000,7.000000,11.000000,5.196152,32.040724,8010.180967,0.010068,0.001143,0.005000,-0.000318,0.001667,0.000211,0.005265,0.000615,0.002896,-0.000345,0.001151,0.000122,-0.001510,0.004902,0.010068,0.001338,0.001016,-0.000318,-0.000915,0.002815,0.005265,0.000936,0.000682,-0.000345,-32.040724,2180.040163,2344.061499,2082.929817,-1579.439311,6028.154905,32.040724,-2180.040163,-2344.061499,-2082.929817,-10600.661529,5299.666069
8,9.000000,9.000000,11.000000,3.000000,-31.975156,-7993.789001,0.002073,0.000048,0.003618,-0.000348,0.00129

### Validation Temperature Loads

In [19]:
filepath = 'inputs/validation_truss_temperature.json'

result = solver.truss_solver(filepath)
pp = postprocess.truss_output(filepath, result)

,node,x (M),y (M),z (M),ux (M),uy (M),uz (M),rx (rad),ry (rad),rz (rad),|u| (M)
0,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-0.000000,-0.000000,0.000000,0.000000
1,2.000000,3.000000,0.000000,0.000000,90.000000,0.000000,0.000000,-0.000000,0.000000,0.000000,90.000000
2,3.000000,0.000000,4.500000,2.598076,-0.000000,135.000000,77.942286,-0.000000,-0.000000,0.000000,155.884573
3,4.000000,3.000000,4.500000,2.598076,90.000000,135.000000,77.942286,-0.000000,-0.000000,0.000000,180.000000
4,5.000000,0.000000,6.000000,0.000000,-0.000000,180.000000,-0.000000,-0.000000,-0.000000,0.000000,180.000000
5,6.000000,3.000000,6.000000,0.000000,90.000000,180.000000,-0.000000,-0.000000,-0.000000,0.000000,201.246118
6,7.000000,0.000000,9.000000,5.196152,-0.000000,270.000000,155.884573,-0.000000,-0.000000,0.000000,311.769145
7,8.000000,3.000000,9.000000,5.196152,90.000000,270.000000,155.884573,-0.000000,-0.000000,0.000000,324.499615
8,9.000000,0.000000,12.000000,0.000000,-0.000000,360.000000,-0.000000,0.000000,-0.000000,0.000000,360.000000
9,10.000000,3.000000,12.000000,0.000000,90.000000,360.000000,0.000000,0.000000,-0.000000,0.000000,371.079506


,metric,node,value (M),ux (M),uy (M),uz (M)
0,max |u|,14.000000,547.448628,90.000000,540.000000,0.000000
1,max |ux|,2.000000,90.000000,90.000000,0.000000,0.000000
2,max |uy|,14.000000,540.000000,90.000000,540.000000,0.000000
3,max |uz|,8.000000,155.884573,90.000000,270.000000,155.884573


,node,Fx (),Fy (),Fz ()
0,1.000000,0.000000,0.000006,0.000002
1,2.000000,nan,-0.000017,-0.000003
2,13.000000,nan,nan,0.000004
3,14.000000,nan,nan,-0.000001


,element,type,i,j,L (M),Δu_local_x (M),axial strain,N_tension (+) (KN),sigma (KN/M^2)
0,1.000000,T,1.000000,5.000000,6.000000,180.000000,30.000000,8400000.000000,2100000000.000089
1,2.000000,T,1.000000,3.000000,5.196152,155.884573,30.000000,8400000.000000,2100000000.000096
2,3.000000,T,3.000000,5.000000,3.000000,90.000000,30.000000,8400000.000000,2100000000.000045
3,4.000000,T,3.000000,7.000000,5.196152,155.884573,30.000000,8400000.000000,2100000000.000039
4,5.000000,T,5.000000,7.000000,6.000000,180.000000,30.000000,8400000.000000,2100000000.000035
5,6.000000,T,5.000000,9.000000,6.000000,180.000000,30.000000,8400000.000000,2100000000.000021
6,7.000000,T,7.000000,9.000000,6.000000,180.000000,30.000000,8400000.000000,2100000000.000001
7,8.000000,T,7.000000,11.000000,5.196152,155.884573,30.000000,8400000.000000,2099999999.999994
8,9.000000,T,9.000000,11.000000,3.000000,90.000000,30.000000,8400000.000000,2099999999.999985
9,10.000000,T,9.000000,13.000000,6.000000,180.000000,30.000000,8400000.000000,2099999999.999948


,element,i,j,L (M),N_tension (+) (KN),sigma (KN/M^2),u_g_1 (M),u_g_2 (M),u_g_3 (M),u_g_4 (rad),u_g_5 (rad),u_g_6 (rad),u_g_7 (M),u_g_8 (M),u_g_9 (M),u_g_10 (rad),u_g_11 (rad),u_g_12 (rad),u_l_1 (M),u_l_2 (M),u_l_3 (M),u_l_4 (rad),u_l_5 (rad),u_l_6 (rad),u_l_7 (M),u_l_8 (M),u_l_9 (M),u_l_10 (rad),u_l_11 (rad),u_l_12 (rad),q_l_1 (KN),q_l_2 (KN),q_l_3 (KN),q_l_4 (KN*M),q_l_5 (KN*M),q_l_6 (KN*M),q_l_7 (KN),q_l_8 (KN),q_l_9 (KN),q_l_10 (KN*M),q_l_11 (KN*M),q_l_12 (KN*M)
0,1.000000,1.000000,5.000000,6.000000,8400000.000000,2100000000.000089,0.000000,0.000000,0.000000,-0.000000,-0.000000,0.000000,-0.000000,180.000000,-0.000000,-0.000000,-0.000000,0.000000,0.000000,0.000000,0.000000,-0.000000,0.000000,0.000000,180.000000,0.000000,-0.000000,-0.000000,0.000000,0.000000,-0.000000,0.000001,-0.000002,0.000000,0.000009,0.000001,0.000000,-0.000001,0.000002,-0.000000,0.000004,0.000006
1,2.000000,1.000000,3.000000,5.196152,8400000.000000,2100000000.000096,0.000000,0.000000,0.000000,-0.000000,-0.000000,0.000000,-0.000000,135.000000,77.942286,-0.000000,-0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-0.000000,0.000000,155.884573,0.000000,0.000000,0.000000,-0.000000,0.000000,-0.000000,-0.000003,0.000001,-0.000000,-0.000001,-0.000009,0.000000,0.000003,-0.000001,0.000000,-0.000004,-0.000004
2,3.000000,3.000000,5.000000,3.000000,8400000.000000,2100000000.000045,-0.000000,135.000000,77.942286,-0.000000,-0.000000,0.000000,-0.000000,180.000000,-0.000000,-0.000000,-0.000000,0.000000,0.000000,155.884573,-0.000000,-0.000000,0.000000,-0.000000,90.000000,155.884573,-0.000000,-0.000000,0.000000,-0.000000,-0.000000,0.000002,-0.000001,0.000001,0.000002,-0.000006,0.000000,-0.000002,0.000001,-0.000001,0.000003,0.000010
3,4.000000,3.000000,7.000000,5.196152,8400000.000000,2100000000.000039,-0.000000,135.000000,77.942286,-0.000000,-0.000000,0.000000,-0.000000,270.000000,155.884573,-0.000000,-0.000000,0.000000,155.884573,0.000000,0.000000,0.000000,-0.000000,0.000000,311.769145,0.000000,0.000000,0.000000,-0.000000,0.000000,-0.000000,0.000001,0.000002,-0.000001,-0.000002,0.000002,0.000000,-0.000001,-0.000002,0.000001,-0.000008,0.000003
4,5.000000,5.000000,7.000000,6.000000,8400000.000000,2100000000.000035,-0.000000,180.000000,-0.000000,-0.000000,-0.000000,0.000000,-0.000000,270.000000,155.884573,-0.000000,-0.000000,0.000000,90.000000,155.884573,0.000000,0.000000,-0.000000,0.000000,270.000000,155.884573,0.000000,0.000000,-0.000000,0.000000,-0.000000,-0.000002,0.000001,-0.000001,-0.000002,-0.000010,0.000000,0.000002,-0.000001,0.000001,-0.000005,-0.000000
5,6.000000,5.000000,9.000000,6.000000,8400000.000000,2100000000.000021,-0.000000,180.000000,-0.000000,-0.000000,-0.000000,0.000000,-0.000000,360.000000,-0.000000,0.000000,-0.000000,0.000000,180.000000,0.000000,-0.000000,-0.000000,0.000000,0.000000,360.000000,0.000000,-0.000000,-0.000000,-0.000000,0.000000,-0.000000,0.000002,-0.000002,0.000000,0.000008,0.000003,0.000000,-0.000002,0.000002,-0.000000,0.000004,0.000010
6,7.000000,7.000000,9.000000,6.000000,8400000.000000,2100000000.000001,-0.000000,270.000000,155.884573,-0.000000,-0.000000,0.000000,-0.000000,360.000000,-0.000000,0.000000,-0.000000,0.000000,0.000000,311.769145,-0.000000,-0.000000,0.000000,-0.000000,180.000000,311.769145,-0.000000,-0.000000,0.000000,0.000000,-0.000000,0.000002,-0.000001,0.000001,0.000002,0.000001,0.000000,-0.000002,0.000001,-0.000001,0.000003,0.000014
7,8.000000,7.000000,11.000000,5.196152,8400000.000000,2099999999.999994,-0.000000,270.000000,155.884573,-0.000000,-0.000000,0.000000,-0.000000,405.000000,77.942286,-0.000000,-0.000000,0.000000,155.884573,270.000000,-0.000000,-0.000000,0.000000,-0.000000,311.769145,270.000000,-0.000000,-0.000000,0.000000,-0.000000,0.000000,0.000002,-0.000001,0.000001,0.000001,0.000002,-0.000000,-0.000002,0.000001,-0.000001,0.000004,0.000006
8,9.000000,9.000000,11.000000,3.000000,8400000.000000,2099999999.999985,-0.000000,360.000000,-0.000000,0.000000,-0.000000,0.000000,-0.000000,405.000000,77.942286,-0.00

### Validation Fabrication Error Loads

In [20]:
filepath = 'inputs/validation_truss_fab_error.json'

result = solver.truss_solver(filepath)
pp = postprocess.truss_output(filepath, result)

,node,x (M),y (M),z (M),ux (M),uy (M),uz (M),rx (rad),ry (rad),rz (rad),|u| (M)
0,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-0.000009,-0.000001,0.000000,0.000000
1,2.000000,3.000000,0.000000,0.000000,0.000222,0.000000,0.000000,-0.000008,0.000001,-0.000005,0.000222
2,3.000000,0.000000,4.500000,2.598076,0.000048,0.000449,0.000169,-0.000013,0.000004,-0.000016,0.000483
3,4.000000,3.000000,4.500000,2.598076,0.000227,0.000422,0.000166,-0.000013,-0.000002,-0.000002,0.000507
4,5.000000,0.000000,6.000000,0.000000,0.000068,0.000540,-0.000007,0.000003,0.000001,-0.000018,0.000545
5,6.000000,3.000000,6.000000,0.000000,0.000234,0.000503,-0.000006,0.000002,-0.000001,-0.000004,0.000555
6,7.000000,0.000000,9.000000,5.196152,0.000126,0.000655,0.000205,-0.000009,0.000000,-0.000012,0.000698
7,8.000000,3.000000,9.000000,5.196152,0.000237,0.000625,0.000204,-0.000009,0.000000,-0.000008,0.000699
8,9.000000,0.000000,12.000000,0.000000,0.000169,0.000731,0.000008,-0.000001,-0.000000,-0.000012,0.000750
9,10.000000,3.000000,12.000000,0.000000,0.000256,0.000700,0.000008,-0.000001,0.000000,-0.000008,0.000746


,metric,node,value (M),ux (M),uy (M),uz (M)
0,max |u|,13.000000,0.000900,0.000240,0.000867,0.000000
1,max |ux|,14.000000,0.000307,0.000307,0.000837,0.000000
2,max |uy|,13.000000,0.000867,0.000240,0.000867,0.000000
3,max |uz|,7.000000,0.000205,0.000126,0.000655,0.000205


,node,Fx (),Fy (),Fz ()
0,1.000000,-0.000000,0.000000,-2.236757
1,2.000000,nan,-0.000000,2.236757
2,13.000000,nan,nan,2.236757
3,14.000000,nan,nan,-2.236757


,element,type,i,j,L (M),Δu_local_x (M),axial strain,N_tension (+) (KN),sigma (KN/M^2)
0,1.000000,T,1.000000,5.000000,6.000000,0.000540,0.000090,25.222109,6305.527160
1,2.000000,T,1.000000,3.000000,5.196152,0.000474,0.000091,25.537864,6384.465950
2,3.000000,T,3.000000,5.000000,3.000000,0.000198,0.000066,18.495534,4623.883444
3,4.000000,T,3.000000,7.000000,5.196152,0.000196,0.000038,10.543430,2635.857375
4,5.000000,T,5.000000,7.000000,6.000000,0.000241,0.000040,11.226193,2806.548349
5,6.000000,T,5.000000,9.000000,6.000000,0.000190,0.000032,8.886527,2221.631796
6,7.000000,T,7.000000,9.000000,6.000000,0.000208,0.000035,9.727648,2431.911909
7,8.000000,T,7.000000,11.000000,5.196152,0.000174,0.000033,9.357775,2339.443699
8,9.000000,T,9.000000,11.000000,3.000000,0.000089,0.000030,8.305356,2076.339020
9,10.000000,T,9.000000,13.000000,6.000000,0.000136,0.000023,6.366731,1591.682756


,element,i,j,L (M),N_tension (+) (KN),sigma (KN/M^2),u_g_1 (M),u_g_2 (M),u_g_3 (M),u_g_4 (rad),u_g_5 (rad),u_g_6 (rad),u_g_7 (M),u_g_8 (M),u_g_9 (M),u_g_10 (rad),u_g_11 (rad),u_g_12 (rad),u_l_1 (M),u_l_2 (M),u_l_3 (M),u_l_4 (rad),u_l_5 (rad),u_l_6 (rad),u_l_7 (M),u_l_8 (M),u_l_9 (M),u_l_10 (rad),u_l_11 (rad),u_l_12 (rad),q_l_1 (KN),q_l_2 (KN),q_l_3 (KN),q_l_4 (KN*M),q_l_5 (KN*M),q_l_6 (KN*M),q_l_7 (KN),q_l_8 (KN),q_l_9 (KN),q_l_10 (KN*M),q_l_11 (KN*M),q_l_12 (KN*M)
0,1.000000,1.000000,5.000000,6.000000,25.222109,6305.527160,0.000000,0.000000,0.000000,-0.000009,-0.000001,0.000000,0.000068,0.000540,-0.000007,0.000003,0.000001,-0.000018,0.000000,0.000000,0.000000,-0.000001,0.000009,0.000000,0.000540,-0.000068,-0.000007,0.000001,-0.000003,-0.000018,441.444558,56.863788,-45.890766,-6.568382,285.669914,384.245719,-441.444558,-56.863788,45.890766,6.568382,-10.325317,-43.062992
1,2.000000,1.000000,3.000000,5.196152,25.537864,6384.465950,0.000000,0.000000,0.000000,-0.000009,-0.000001,0.000000,0.000048,0.000449,0.000169,-0.000013,0.000004,-0.000016,0.000000,0.000000,0.000000,-0.000000,-0.000001,0.000009,0.000474,0.000078,-0.000048,-0.000005,0.000015,0.000013,-25.537864,-118.899669,57.735782,22.142150,-364.399226,-355.755853,25.537864,118.899669,-57.735782,-22.142150,64.395302,-262.064951
2,3.000000,3.000000,5.000000,3.000000,18.495534,4623.883444,0.000048,0.000449,0.000169,-0.000013,0.000004,-0.000016,0.000068,0.000540,-0.000007,0.000003,0.000001,-0.000018,0.000078,0.000474,0.000048,0.000015,-0.000005,-0.000013,0.000276,0.000465,0.000068,0.000016,-0.000008,0.000003,-18.495534,-163.539140,-10.181229,-6.073476,99.633138,-622.442641,18.495534,163.539140,10.181229,6.073476,-69.089450,131.825222
3,4.000000,3.000000,7.000000,5.196152,10.543430,2635.857375,0.000048,0.000449,0.000169,-0.000013,0.000004,-0.000016,0.000126,0.000655,0.000205,-0.000009,0.000000,-0.000012,0.000474,0.000078,-0.000048,-0.000005,0.000015,0.000013,0.000670,0.000150,-0.000126,-0.000006,0.000011,0.000009,-10.543430,-91.320766,57.427107,7.774648,-88.466977,-183.282407,10.543430,91.320766,-57.427107,-7.774648,-209.933022,-291.234211
4,5.000000,5.000000,7.000000,6.000000,11.226193,2806.548349,0.000068,0.000540,-0.000007,0.000003,0.000001,-0.000018,0.000126,0.000655,0.000205,-0.000009,0.000000,-0.000012,0.000264,0.000472,-0.000068,-0.000015,0.000010,-0.000003,0.000505,0.000465,-0.000126,-0.000011,0.000006,0.000009,-11.226193,92.688762,37.942582,-19.851876,-73.777626,136.243823,11.226193,-92.688762,-37.942582,19.851876,-153.877864,419.888750
5,6.000000,5.000000,9.000000,6.000000,8.886527,2221.631796,0.000068,0.000540,-0.000007,0.000003,0.000001,-0.000018,0.000169,0.000731,0.000008,-0.000001,-0.000000,-0.000012,0.000540,-0.000068,-0.000007,0.000001,-0.000003,-0.000018,0.000731,-0.000169,0.000008,-0.000000,0.000001,-0.000012,-8.886527,42.667465,-26.155386,5.186822,34.807461,58.860593,8.886527,-42.667465,26.155386,-5.186822,122.124855,197.144199
6,7.000000,7.000000,9.000000,6.000000,9.727648,2431.911909,0.000126,0.000655,0.000205,-0.000009,0.000000,-0.000012,0.000169,0.000731,0.000008,-0.000001,-0.000000,-0.000012,0.000150,0.000670,0.000126,0.000011,-0.000006,-0.000009,0.000359,0.000637,0.000169,0.000010,-0.000006,-0.000001,-9.727648,16.688210,-23.199846,2.357526,71.382363,-48.099135,9.727648,-16.688210,23.199846,-2.357526,67.816714,148.228397
7,8.000000,7.000000,11.000000,5.196152,9.357775,2339.443699,0.000126,0.000655,0.000205,-0.000009,0.000000,-0.000012,0.000186,0.000784,0.000080,-0.000006,-0.000001,-0.000012,0.000465,0.000505,0.000126,0.000006,-0.000011,-0.000009,0.000639,0.000461,0.000186,0.000005,-0.000011,-0.000006,-9.357775,25.414573,-24.793244,5.763470,65.079307,28.955660,9.357775,-25.414573,24.793244,-5.763470,63.750166,103.102332
8,9.000000,9.000000,11.000000,3.000000,8.305356,2076.339020,0.000169,0.000731,0.000008,-0.000001,-0.000000,-0.000012,0.000186,0.000784,0.000080,-0.000006,-0.000001,-0.000012,0.000372,0.000629,-0.000169,-0.000011,0.000006,0.000001,0.000461,0.0